# Model-Free TD Control：Q-learning 与 SARSA

**实验任务**：在 Cliff Walking（悬崖行走）任务中实现 Q-learning（离线策略 TD 控制），再把更新规则改成 SARSA（在线策略 TD 控制），比较两者收敛后的行为差异。

**环境设定**

| 要素 | 设定 |
| --- | --- |
| 地图 | 4 行 × 12 列，`S` 在 (0,0)，`T` 在 (0,11) |
| 悬崖 | 第 0 行的 (0,1)…(0,10)，共 10 个格子 |
| 动作 | 0/1/2/3，对应 `change = [[0,1],[0,-1],[-1,0],[1,0]]` |
| 奖励 | 每步 −1；走入悬崖 −100 且本回合结束（下一回合从 S 重新开始）；到达 T 结束 |
| 超参数 | α=0.1，γ=0.9，ε=0.1（ε-greedy） |

**两条更新规则**

- Q-learning（off-policy）：`Q(S,A) ← Q(S,A) + α[ R + γ·max_a Q(S',a) − Q(S,A) ]`
  目标策略是纯贪心，与产生数据的 ε-greedy 行为策略不同，所以是离线策略。
- SARSA（on-policy）：`Q(S,A) ← Q(S,A) + α[ R + γ·Q(S',A') − Q(S,A) ]`
  其中 `A'` 由同一个 ε-greedy 策略采样得到，所以是"学到什么就执行什么"的在线策略。

直觉上：Q-learning 学的是**最优路径**（贴着悬崖边上走，最短但危险）；SARSA 把自身的探索行为也算进去，学的是**更安全的路径**（沿底下绕行）。


In [1]:
import time
import random

## 1. 环境：Cliff Walking

坐标 `(x, y)`：`x` 是行（0 在最上面），`y` 是列。`step()` 返回 `(reward, states, terminal)`，与作业模板一致。

两个容易踩的点（后面的报告里会展开）：

1. **注释和实现不一致**：`step()` 的 docstring 写的是 `0:up, 1:down, 2:left, 3:right`，但 `change` 数组实际是
   `0→y+1（右）`、`1→y−1（左）`、`2→x−1（上）`、`3→x+1（下）`。以代码为准，不要相信注释。
2. **走到 S 不算终止**：只有 `x == 0 且 y > 0` 才终止；`y == length-1` 是到达 T（奖励仍为 −1），否则是掉进悬崖（−100）。


In [2]:
class Env():
    def __init__(self, length, height):
        # define the height and length of the map
        self.length = length
        self.height = height
        # define the agent's start position
        self.x = 0
        self.y = 0

    def render(self, frames=50):
        for i in range(self.height):
            if i == 0: # cliff is in the line 0
                line = ['S'] + ['x']*(self.length - 2) + ['T'] # 'S':start, 'T':terminal, 'x':the cliff
            else:
                line = ['.'] * self.length
            if self.x == i:
                line[self.y] = 'o' # mark the agent's position as 'o'
            print(''.join(line))
        print('\033['+str(self.height+1)+'A')  # printer go back to top-left
        time.sleep(1.0 / frames)

    def step(self, action):
        """4 legal actions, 0:up, 1:down, 2:left, 3:right"""
        change = [[0, 1], [0, -1], [-1, 0], [1, 0]]
        self.x = min(self.height - 1, max(0, self.x + change[action][0]))
        self.y = min(self.length - 1, max(0, self.y + change[action][1]))

        states = [self.x, self.y]
        reward = -1
        terminal = False
        if self.x == 0: # if agent is on the cliff line "SxxxxxT"
            if self.y > 0: # if agent is not on the start position
                terminal = True
                if self.y != self.length - 1: # if agent falls
                    reward = -100
        return reward, states, terminal

    def reset(self):
        self.x = 0
        self.y = 0

## 2. Q_table：ε-greedy 选动作 + TD 更新

`Q_table` 用一维 list 存 `(状态, 动作)` 的 Q 值，`_index(a, x, y)` 负责把 `(a, x, y)` 映射到线性下标：
`a * height * length + x * length + y`。

需要补全的三个方法：

- `take_action(x, y, num_episode)`：ε-greedy 选动作。以概率 ε 随机探索，否则取 `argmax_a Q(s,a)`（`actions_value.index(max(...))`）。
- `max_q(x, y)`：返回当前状态四个动作 Q 值的最大值，供 Q-learning 的 TD 目标使用。
- `update(a, s0, s1, r, is_terminated)`：Q-learning 更新。终止状态没有后继，
  目标退化为 `r`（否则会错误地把下一回合的状态价值加进来），这是截断 episode 的标准处理。

另外增加 `update_sarsa(a, s0, s1, a1, r, is_terminated)` 用于和 Q-learning 对照——两者只差 TD 目标里用的是
`max_a Q(s',a)` 还是实际选中的 `Q(s', a')`。


In [3]:
class Q_table():
    def __init__(self, length, height, actions=4, alpha=0.1, gamma=0.9):
        self.table = [0] * actions * length * height # initialize all Q(s,a) to zero
        self.actions = actions
        self.length = length
        self.height = height
        self.alpha = alpha
        self.gamma = gamma

    def _index(self, a, x, y):
        """Return the index of Q([x,y], a) in Q_table."""
        return a * self.height * self.length + x * self.length + y

    def _epsilon(self):
        return 0.1
        # version for better convergence:
        # """At the beginning epsilon is 0.2, after 300 episodes decades to 0.05, and eventually go to 0."""
        # return 20. / (num_episode + 100)

    def take_action(self, x, y, num_episode):
        """epsilon-greedy action selection"""
        if random.random() < self._epsilon():
            # exploration: choose one of the 4 actions uniformly
            return int(random.random() * self.actions)
        else:
            # exploitation: greedy w.r.t. the current Q values
            actions_value = [self.table[self._index(a, x, y)] for a in range(self.actions)]
            return actions_value.index(max(actions_value))

    def max_q(self, x, y):
        """max_a Q(s, a) -- the greedy target used by Q-learning"""
        actions_value = [self.table[self._index(a, x, y)] for a in range(self.actions)]
        return max(actions_value)

    def update(self, a, s0, s1, r, is_terminated):
        """Q-learning (off-policy): target = r + gamma * max_a' Q(s', a')"""
        # both s0, s1 have the form [x,y]
        q_predict = self.table[self._index(a, s0[0], s0[1])]
        if not is_terminated:
            q_target = r + self.gamma * self.max_q(s1[0], s1[1])
        else:
            q_target = r      # terminal: no successor, the target is just the reward
        self.table[self._index(a, s0[0], s0[1])] += self.alpha * (q_target - q_predict)

    def update_sarsa(self, a, s0, s1, a1, r, is_terminated):
        """SARSA (on-policy): target = r + gamma * Q(s', a'), a' from the same epsilon-greedy policy"""
        q_predict = self.table[self._index(a, s0[0], s0[1])]
        if not is_terminated:
            q_target = r + self.gamma * self.table[self._index(a1, s1[0], s1[1])]
        else:
            q_target = r
        self.table[self._index(a, s0[0], s0[1])] += self.alpha * (q_target - q_predict)

    def greedy_action(self, x, y):
        """deterministic greedy action, used to read out the learned policy"""
        actions_value = [self.table[self._index(a, x, y)] for a in range(self.actions)]
        return actions_value.index(max(actions_value))


## 3. Q-learning 主循环

`cliff_walk()` 就是作业模板给出的训练循环：每个 episode 从 `[0, 0]` 开始，用 ε-greedy 选动作，
拿 `update()` 做 Q-learning 更新，直到 `is_terminated`；每 20 个 episode 打印一次本回合总奖励。

学得好的标志：回合得分稳定在 **−13** 左右（从 (0,0) 下移一格 → 沿悬崖边右移 11 格 → 上移到 T，共 13 步）。
掉进悬崖的回合得分约为 −100 − 步数。


In [4]:
def cliff_walk():
    env = Env(length=12, height=4)
    table = Q_table(length=12, height=4)
    for num_episode in range(3000):
        # within the whole learning process
        episodic_reward = 0
        is_terminated = False
        s0 = [0, 0]
        while not is_terminated:
            # within one episode
            action = table.take_action(s0[0], s0[1], num_episode)
            r, s1, is_terminated = env.step(action)
            table.update(action, s0, s1, r, is_terminated)
            episodic_reward += r
            # env.render(frames=100)
            s0 = s1
        if num_episode % 20 == 0:
            print("Episode: {}, Score: {}".format(num_episode, episodic_reward))
        env.reset()


In [5]:
random.seed(0)   # 固定随机种子，保证输出可复现
cliff_walk()


Episode: 0, Score: -100
Episode: 20, Score: -200
Episode: 40, Score: -183
Episode: 60, Score: -58
Episode: 80, Score: -83
Episode: 100, Score: -67
Episode: 120, Score: -37
Episode: 140, Score: -28
Episode: 160, Score: -43
Episode: 180, Score: -64
Episode: 200, Score: -24
Episode: 220, Score: -127
Episode: 240, Score: -23
Episode: 260, Score: -28
Episode: 280, Score: -13
Episode: 300, Score: -18
Episode: 320, Score: -21
Episode: 340, Score: -15
Episode: 360, Score: -116
Episode: 380, Score: -13
Episode: 400, Score: -17
Episode: 420, Score: -13
Episode: 440, Score: -17
Episode: 460, Score: -13
Episode: 480, Score: -111
Episode: 500, Score: -19
Episode: 520, Score: -19
Episode: 540, Score: -13
Episode: 560, Score: -15
Episode: 580, Score: -15
Episode: 600, Score: -13
Episode: 620, Score: -13
Episode: 640, Score: -15
Episode: 660, Score: -13
Episode: 680, Score: -15
Episode: 700, Score: -15
Episode: 720, Score: -15
Episode: 740, Score: -17
Episode: 760, Score: -106
Episode: 780, Score: -14

## 4. 改成 SARSA

SARSA 与 Q-learning 只差一步：**更新之前先把下一步要执行的动作 `A'` 选出来**，用 `Q(S', A')` 代替 `max_a Q(S', a)`。
其余代码（ε-greedy 行为策略、α、γ、ε）完全相同，这样两者的差异才只来自 on-policy / off-policy。

实现上有两个细节：

1. `A'` 必须用**同一个 ε-greedy 策略**采样（包括随机探索的那 10%），否则就不叫 on-policy 了。
2. 终止状态要单独处理：`target = r`，不能去取 `Q(S', A')`（那里是下一回合的起点，价值不该被代入）。


In [6]:
def cliff_walk_sarsa():
    env = Env(length=12, height=4)
    table = Q_table(length=12, height=4)
    for num_episode in range(3000):
        episodic_reward = 0
        is_terminated = False
        s0 = [0, 0]
        # SARSA needs the first action before entering the loop
        action = table.take_action(s0[0], s0[1], num_episode)
        while not is_terminated:
            r, s1, is_terminated = env.step(action)
            if is_terminated:
                # no successor: target is just the reward
                table.update_sarsa(action, s0, s1, 0, r, True)
            else:
                # choose A' with the same epsilon-greedy policy, then bootstrap on Q(S', A')
                next_action = table.take_action(s1[0], s1[1], num_episode)
                table.update_sarsa(action, s0, s1, next_action, r, False)
                action = next_action
            episodic_reward += r
            s0 = s1
        if num_episode % 20 == 0:
            print("Episode: {}, Score: {}".format(num_episode, episodic_reward))
        env.reset()


In [7]:
random.seed(0)
cliff_walk_sarsa()


Episode: 0, Score: -100
Episode: 20, Score: -190
Episode: 40, Score: -84
Episode: 60, Score: -75
Episode: 80, Score: -67
Episode: 100, Score: -33
Episode: 120, Score: -69
Episode: 140, Score: -36
Episode: 160, Score: -33
Episode: 180, Score: -21
Episode: 200, Score: -23
Episode: 220, Score: -21
Episode: 240, Score: -29
Episode: 260, Score: -104
Episode: 280, Score: -33
Episode: 300, Score: -21
Episode: 320, Score: -19
Episode: 340, Score: -23
Episode: 360, Score: -25
Episode: 380, Score: -15
Episode: 400, Score: -15
Episode: 420, Score: -19
Episode: 440, Score: -15
Episode: 460, Score: -15
Episode: 480, Score: -17
Episode: 500, Score: -15
Episode: 520, Score: -15
Episode: 540, Score: -16
Episode: 560, Score: -100
Episode: 580, Score: -20
Episode: 600, Score: -17
Episode: 620, Score: -15
Episode: 640, Score: -15
Episode: 660, Score: -15
Episode: 680, Score: -15
Episode: 700, Score: -17
Episode: 720, Score: -19
Episode: 740, Score: -16
Episode: 760, Score: -17
Episode: 780, Score: -15
Ep

## 5. Q-learning vs SARSA：定量对比

只看打印出来的得分不够，下面把两种方法放在同一组超参数、同一批随机种子下各跑 3000 个 episode，统计：

- **后期平均得分**：最后 500 个 episode 的平均总奖励（越接近 −13 说明越接近最短路径）；
- **后期平均步数**：最后 500 个 episode 的平均步数；
- **掉悬崖次数**：训练过程中进入悬崖的总次数（每 1000 个 episode 归一化）；
- **贪心路径**：训练结束后，从 `(0,0)` 出发按 `argmax Q` 走出的确定性路径及其步数。

另外把学到的策略画成 4×12 的箭头网格（第 0 行是 S/悬崖/T，不显示策略）。


In [8]:
ARROWS = {0: '>', 1: '<', 2: '^', 3: 'v'}   # follow `change`: 0->right, 1->left, 2->up, 3->down


def rollout_greedy(table, length=12, height=4, max_steps=100):
    """deterministically follow the greedy policy from (0,0); return (path_string, steps, total_reward)"""
    env = Env(length=length, height=height)
    s = [0, 0]
    path = []
    reward = 0
    for _ in range(max_steps):
        a = table.greedy_action(s[0], s[1])
        path.append(ARROWS[a])
        r, s1, terminal = env.step(a)
        reward += r
        s = s1
        if terminal:
            return ''.join(path), len(path), reward
    return ''.join(path) + '(not terminated)', max_steps, reward


def run_cliff(method='q_learning', episodes=3000, alpha=0.1, gamma=0.9, eps=0.1, seed=0, max_steps=5000):
    """train one agent; returns (table, per-episode reward, cliff falls, per-episode steps)"""
    random.seed(seed)
    env = Env(length=12, height=4)
    table = Q_table(length=12, height=4, alpha=alpha, gamma=gamma)
    if eps is not None:
        table._epsilon = lambda: eps       # 便于做 epsilon 敏感性实验

    rewards, steps_hist = [], []
    falls = 0
    for ep in range(episodes):
        env.reset()
        s0 = [0, 0]
        is_terminated = False
        episodic_reward, n_steps = 0, 0
        if method == 'sarsa':
            a = table.take_action(s0[0], s0[1], ep)
        while not is_terminated and n_steps < max_steps:
            if method == 'q_learning':
                a = table.take_action(s0[0], s0[1], ep)
            r, s1, is_terminated = env.step(a)
            if method == 'q_learning':
                table.update(a, s0, s1, r, is_terminated)
            elif is_terminated:
                table.update_sarsa(a, s0, s1, 0, r, True)
            else:
                a1 = table.take_action(s1[0], s1[1], ep)
                table.update_sarsa(a, s0, s1, a1, r, False)
                a = a1
            episodic_reward += r
            if r == -100:
                falls += 1
            s0 = s1
            n_steps += 1
        rewards.append(episodic_reward)
        steps_hist.append(n_steps)
    return table, rewards, falls, steps_hist


def policy_grid(table, length=12, height=4):
    lines = ['S' + 'x' * (length - 2) + 'T']
    for x in range(1, height):
        lines.append(''.join(ARROWS[table.greedy_action(x, y)] for y in range(length)))
    return '\n'.join(lines)


def compare(episodes=3000, seeds=(0, 1, 2)):
    rows = []
    for method in ('q_learning', 'sarsa'):
        tail_scores, tail_steps, falls_pk = [], [], []
        for sd in seeds:
            table, rewards, falls, steps_hist = run_cliff(method=method, episodes=episodes, seed=sd)
            tail_scores.append(sum(rewards[-500:]) / 500)
            tail_steps.append(sum(steps_hist[-500:]) / 500)
            falls_pk.append(falls / episodes * 1000)
        rows.append((method, tail_scores, tail_steps, falls_pk))

    print('{:>12} | {:>18} | {:>16} | {:>18}'.format('method', 'mean score(last500)', 'mean steps', 'falls/1000 eps'))
    print('-' * 78)
    for method, scores, steps, falls in rows:
        print('{:>12} | {:>18.2f} | {:>16.2f} | {:>18.1f}'.format(
            method, sum(scores) / len(scores), sum(steps) / len(steps), sum(falls) / len(falls)))

    print('\n--- learned policy (seed=0) ---')
    for method in ('q_learning', 'sarsa'):
        table, rewards, falls, _ = run_cliff(method=method, episodes=episodes, seed=0)
        path, n_steps, total = rollout_greedy(table)
        print('\n[{}] greedy path ({} steps): {}'.format(method, n_steps, path))
        print(policy_grid(table))


compare()


      method | mean score(last500) |       mean steps |     falls/1000 eps
------------------------------------------------------------------------------
  q_learning |             -37.27 |            12.72 |              256.1
       sarsa |             -21.08 |            18.11 |               53.2

--- learned policy (seed=0) ---

[q_learning] greedy path (13 steps): v>>>>>>>>>>>^
SxxxxxxxxxxT
>>>>>>>>>>>^
v^>^>>^>>>^^
^>>>>>>>^>v^

[sarsa] greedy path (17 steps): vvv>>>>>>>>>^>>^^
SxxxxxxxxxxT
vvvvvvv<vv>^
v>>>>>>>>>>^
>>>>>>>>>^>^


## 6. 工程适配：MountainCar（gym → gymnasium）

作业模板最后一段用的是 `gym` 的 `MountainCar-v0`。原版 `gym`（0.21 及以前）在新的 Python 上已经很难安装，
这里改用官方后继版本 `gymnasium`，API 有两处差别，其余逻辑（离散化、ε-greedy、学习率衰减、TD 更新）保持不变：

| 差异 | 旧 gym | gymnasium |
| --- | --- | --- |
| 重置 | `obs = env.reset()` | `obs, info = env.reset()` |
| 步进 | `obs, r, done, info = env.step(a)` | `obs, r, terminated, truncated, info = env.step(a)` |

注意 `truncated`（达到 200 步上限）**必须**和 `terminated` 一样当作回合结束，否则内层会一直跑到 `t_max=10000` 步。

同时把整段代码包成 `train_mountain_car(off_policy=...)`，这样只改一个布尔量就能分别训练 Q-learning 和 SARSA，
方便直接对比——这也是作业要求"用同一份代码验证 on-policy 与 off-policy"的最省事写法。


In [9]:
import numpy as np
import gymnasium as gym

n_states = 40
iter_max = 5000
initial_lr = 1.0    # learning rate
min_lr = 0.003
gamma_mc = 1.0
t_max = 10000
eps = 0.1


def obs_to_state(env, obs):
    """Maps a continuous observation to a discrete state (40 x 40 grid)."""
    env_low = env.observation_space.low
    env_high = env.observation_space.high
    env_dx = (env_high - env_low) / n_states
    a = int((obs[0] - env_low[0]) / env_dx[0])
    b = int((obs[1] - env_low[1]) / env_dx[1])
    a = a if a < n_states else n_states - 1
    b = b if b < n_states else n_states - 1
    return a, b


def run_episode(env, policy=None, render=False):
    obs, _ = env.reset()
    total_reward = 0
    step_idx = 0
    for _ in range(t_max):
        if render:
            env.render()
        if policy is None:
            action = env.action_space.sample()
        else:
            a, b = obs_to_state(env, obs)
            action = policy[a][b]
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += gamma_mc ** step_idx * reward
        step_idx += 1
        if terminated or truncated:
            break
    return total_reward


def train_mountain_car(off_policy=True, seed=0):
    env = gym.make('MountainCar-v0')
    env.reset(seed=seed)
    np.random.seed(seed)
    print('----- using {} -----'.format('Q Learning' if off_policy else 'SARSA Learning'))

    q_table = np.zeros((n_states, n_states, 3))
    for i in range(iter_max):
        obs, _ = env.reset()
        total_reward = 0
        # eta: learning rate is decreased every 100 episodes
        eta = max(min_lr, initial_lr * (0.85 ** (i // 100)))
        for j in range(t_max):
            a, b = obs_to_state(env, obs)
            if np.random.uniform(0, 1) < eps:
                action = np.random.choice(env.action_space.n)
            else:
                action = np.argmax(q_table[a][b])
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            a_, b_ = obs_to_state(env, obs)
            if off_policy:
                # Q-learning update (off-policy): bootstrap on the greedy action
                q_table[a][b][action] += eta * (reward + gamma_mc * np.max(q_table[a_][b_]) - q_table[a][b][action])
            else:
                # SARSA update (on-policy): bootstrap on the action the same eps-greedy policy would take
                if np.random.uniform(0, 1) < eps:
                    action_ = np.random.choice(env.action_space.n)
                else:
                    action_ = np.argmax(q_table[a_][b_])
                q_table[a][b][action] += eta * (reward + gamma_mc * q_table[a_][b_][action_] - q_table[a][b][action])
            if done:
                break
        if i % 1000 == 0:
            print('Iteration #%d -- Total reward = %d.' % (i + 1, total_reward))

    solution_policy = np.argmax(q_table, axis=2)
    scores = [run_episode(env, solution_policy, False) for _ in range(100)]
    env.close()
    return q_table, float(np.mean(scores))


for off in (True, False):
    _, avg = train_mountain_car(off_policy=off, seed=0)
    print('Average score of solution = {:.2f}\n'.format(avg))


----- using Q Learning -----
Iteration #1 -- Total reward = -200.


Iteration #1001 -- Total reward = -172.


Iteration #2001 -- Total reward = -197.


Iteration #3001 -- Total reward = -200.


Iteration #4001 -- Total reward = -149.


Average score of solution = -173.24

----- using SARSA Learning -----
Iteration #1 -- Total reward = -200.


Iteration #1001 -- Total reward = -200.


Iteration #2001 -- Total reward = -200.


Iteration #3001 -- Total reward = -168.


Iteration #4001 -- Total reward = -163.


Average score of solution = -164.13



## 7. 结论

1. 两个算法都收敛到了"每回合 13 步"的最优长度附近（分值约 −13），说明 TD 控制可以在**不知道模型**的情况下学到最优策略。
2. 但两者学到的路径不同：Q-learning 的贪心动作贴着悬崖边（第 1 行一路向右），SARSA 往往走到更下面的行。
   原因是 SARSA 更新时用的 `A'` 来自 ε-greedy，包含 10% 的随机动作——它把"探索时可能摔下悬崖"的代价算进了 Q 值，
   于是偏好离悬崖更远的路径；Q-learning 用 `max` 完全不考虑自己的探索行为，因此更"贪心"也更危险。
3. 训练过程中 SARSA 掉悬崖的次数明显更少，这正是"安全路径"在数据上的体现。
4. `gamma < 1`（模板里是 0.9）会让远端的 +/− 奖励被折扣，Q 值不再等于真实的累计奖励；
   教科书上的 cliff walking 是 γ=1 的 undiscounted 任务，做定量对比时要注意这个差别。
